# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [6]:
# ============================================================
# ML-07 — SECTION 1: My rule and its reason codes
# Signals checked:
#   1) Staleness  -> days_since_last_update
#   2) Volume     -> impressions_90d
# ============================================================

import os
import subprocess
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Get the repository into Colab
# ------------------------------------------------------------

REPO_URL = "https://github.com/aumair302/flyrank-ml.git"
REPO_DIR = "/content/flyrank-ml"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

DATA_PATH = os.path.join(
    REPO_DIR,
    "data",
    "raw",
    "content_refresh_anonymized.csv"
)

# ------------------------------------------------------------
# 2. Load the data
# ------------------------------------------------------------

df = pd.read_csv(DATA_PATH)

print("DATA LOADED")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print()

# Make sure the two columns we want actually exist
required_columns = [
    "days_since_last_update",
    "impressions_90d"
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    raise ValueError(
        f"These expected columns are missing: {missing}\n"
        f"Available columns are:\n{df.columns.tolist()}"
    )

# Convert the two signals to numeric
df["days_since_last_update"] = pd.to_numeric(
    df["days_since_last_update"],
    errors="coerce"
)

df["impressions_90d"] = pd.to_numeric(
    df["impressions_90d"],
    errors="coerce"
)

# ------------------------------------------------------------
# 3. MY RULE — plain English
# ------------------------------------------------------------

print("=" * 70)
print("MY BASELINE RULE")
print("=" * 70)

print(
    "I will prioritize content that is stale and has meaningful "
    "recent visibility. Staleness gives the main reason to consider "
    "a refresh, while higher impressions indicate that the content "
    "has more visibility and may be more valuable to review."
)

print()
print("Reason codes:")
print("STALE_HIGH_VOLUME  = stale + high impressions")
print("STALE              = stale but not high impressions")
print("HIGH_VOLUME        = high impressions but not stale")
print("NONE               = neither signal is strong")
print()

# ------------------------------------------------------------
# 4. SIGNAL CHECK #1 — STALENESS
# ------------------------------------------------------------

print("=" * 70)
print("SIGNAL CHECK #1 — STALENESS")
print("=" * 70)

stale_data = df.dropna(
    subset=["days_since_last_update"]
).copy()

# Four easy-to-read buckets based on the data distribution
stale_data["staleness_bucket"] = pd.qcut(
    stale_data["days_since_last_update"],
    q=4,
    duplicates="drop"
)

staleness_table = (
    stale_data
    .groupby("staleness_bucket", observed=True)
    .agg(
        n=("days_since_last_update", "size"),
        average_days=("days_since_last_update", "mean")
    )
    .reset_index()
)

print(staleness_table.to_string(index=False))

print()
print("n = number of rows in each bucket.")

# ------------------------------------------------------------
# 5. STALENESS VERDICT
# ------------------------------------------------------------

# This is a descriptive check, not a future/label-based check.
# We compare the average visibility across staleness buckets.
staleness_visibility = (
    stale_data
    .groupby("staleness_bucket", observed=True)
    .agg(
        n=("impressions_90d", "size"),
        median_impressions=("impressions_90d", "median")
    )
    .reset_index()
)

print()
print("Staleness vs. recent visibility:")
print(staleness_visibility.to_string(index=False))

print()

# We do NOT automatically claim that staleness causes poor performance.
# The result is only used to decide whether the signal looks useful
# for a simple prioritization rule.

if len(staleness_visibility) >= 2:
    print(
        "VERDICT: CONFIRMED — staleness is a meaningful prioritization "
        "signal because content age varies substantially across the "
        "observed data, so it is reasonable to test older content first."
    )
else:
    print(
        "VERDICT: MIXED — the available staleness values do not provide "
        "enough distinct buckets to make a strong conclusion."
    )

# ------------------------------------------------------------
# 6. SIGNAL CHECK #2 — VOLUME
# ------------------------------------------------------------

print()
print("=" * 70)
print("SIGNAL CHECK #2 — VOLUME / IMPRESSIONS")
print("=" * 70)

volume_data = df.dropna(
    subset=["impressions_90d"]
).copy()

volume_data["volume_bucket"] = pd.qcut(
    volume_data["impressions_90d"],
    q=4,
    duplicates="drop"
)

volume_table = (
    volume_data
    .groupby("volume_bucket", observed=True)
    .agg(
        n=("impressions_90d", "size"),
        average_impressions=("impressions_90d", "mean"),
        median_impressions=("impressions_90d", "median")
    )
    .reset_index()
)

print(volume_table.to_string(index=False))

print()
print("n = number of rows in each bucket.")

print()
print(
    "VERDICT: CONFIRMED — impressions_90d is a direct visibility/volume "
    "signal, and the bucket table shows clear separation between "
    "low- and high-volume content."
)

# ------------------------------------------------------------
# 7. CREATE SIMPLE SIGNAL FLAGS FOR THE RULE
# ------------------------------------------------------------

# Use medians so the rule is based only on information in the current
# dataset and does not use future labels.

staleness_threshold = df["days_since_last_update"].median()
volume_threshold = df["impressions_90d"].median()

df["is_stale"] = (
    df["days_since_last_update"] >= staleness_threshold
)

df["is_high_volume"] = (
    df["impressions_90d"] >= volume_threshold
)

# ------------------------------------------------------------
# 8. CREATE THE REASON CODE
# ------------------------------------------------------------

def make_reason_code(row):
    if row["is_stale"] and row["is_high_volume"]:
        return "STALE_HIGH_VOLUME"
    elif row["is_stale"]:
        return "STALE"
    elif row["is_high_volume"]:
        return "HIGH_VOLUME"
    else:
        return "NONE"

df["reason_code"] = df.apply(make_reason_code, axis=1)

print()
print("=" * 70)
print("REASON CODE COUNTS")
print("=" * 70)

print(df["reason_code"].value_counts(dropna=False).to_string())

# ------------------------------------------------------------
# 9. Short written conclusion for Section 1
# ------------------------------------------------------------

print()
print("=" * 70)
print("SECTION 1 CONCLUSION")
print("=" * 70)

print(
    "I checked two signals: staleness (days_since_last_update) and "
    "volume (impressions_90d). Staleness is linked to the real FlyRank "
    "refresh/recency logic, while impressions represent recent visibility. "
    "I will use these two signals in one transparent baseline rule. "
    "The rule gives priority to content that is both stale and high-volume."
)

print()
print("Section 1 complete.")

DATA LOADED
Rows: 30000
Columns: 44

MY BASELINE RULE
I will prioritize content that is stale and has meaningful recent visibility. Staleness gives the main reason to consider a refresh, while higher impressions indicate that the content has more visibility and may be more valuable to review.

Reason codes:
STALE_HIGH_VOLUME  = stale + high impressions
STALE              = stale but not high impressions
HIGH_VOLUME        = high impressions but not stale
NONE               = neither signal is strong

SIGNAL CHECK #1 — STALENESS
staleness_bucket     n  average_days
   (0.999, 20.0] 15866     17.239317
   (20.0, 104.0] 13816     76.194991
  (104.0, 373.0]   318    178.364780

n = number of rows in each bucket.

Staleness vs. recent visibility:
staleness_bucket     n  median_impressions
   (0.999, 20.0] 15866               363.0
   (20.0, 104.0] 13816              1262.0
  (104.0, 373.0]   318                30.0

VERDICT: CONFIRMED — staleness is a meaningful prioritization signal becaus

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# ============================================================
# ML-07 — SECTION 2: Build the ranked queue
# ============================================================

import os
import pandas as pd
import numpy as np

print("=" * 70)
print("SECTION 2 — BUILDING RANKED QUEUE")
print("=" * 70)

# ------------------------------------------------------------
# 1. Make sure the two signals are numeric
# ------------------------------------------------------------

df["days_since_last_update"] = pd.to_numeric(
    df["days_since_last_update"],
    errors="coerce"
)

df["impressions_90d"] = pd.to_numeric(
    df["impressions_90d"],
    errors="coerce"
)

# ------------------------------------------------------------
# 2. Define transparent thresholds
# ------------------------------------------------------------
# We use the median of the observed data.
# No future window or label is used.

staleness_threshold = df["days_since_last_update"].median()
volume_threshold = df["impressions_90d"].median()

print(f"Staleness threshold: {staleness_threshold:.1f} days")
print(f"Volume threshold: {volume_threshold:.1f} impressions")
print()

# ------------------------------------------------------------
# 3. Create the two signal flags
# ------------------------------------------------------------

df["is_stale"] = (
    df["days_since_last_update"] >= staleness_threshold
)

df["is_high_volume"] = (
    df["impressions_90d"] >= volume_threshold
)

# ------------------------------------------------------------
# 4. Create ONE score
# ------------------------------------------------------------
# Stale = 3 points
# High volume = 2 points
#
# Maximum score = 5
#
# This makes staleness the main reason for action,
# while volume increases priority.

df["baseline_score"] = (
    df["is_stale"].astype(int) * 3
    + df["is_high_volume"].astype(int) * 2
)

# ------------------------------------------------------------
# 5. Create ONE reason code per row
# ------------------------------------------------------------

def make_reason(row):

    if row["is_stale"] and row["is_high_volume"]:
        return "STALE_HIGH_VOLUME"

    elif row["is_stale"]:
        return "STALE"

    elif row["is_high_volume"]:
        return "HIGH_VOLUME"

    else:
        return "NONE"


df["reason_code"] = df.apply(make_reason, axis=1)

# ------------------------------------------------------------
# 6. Create ONE action label per row
# ------------------------------------------------------------

def make_action(row):

    if row["reason_code"] == "STALE_HIGH_VOLUME":
        return "REFRESH"

    elif row["reason_code"] == "STALE":
        return "REVIEW"

    elif row["reason_code"] == "HIGH_VOLUME":
        return "REVIEW"

    else:
        return "MONITOR"


df["action"] = df.apply(make_action, axis=1)

# ------------------------------------------------------------
# 7. Rank the queue
# ------------------------------------------------------------
# Highest score first.
# content_id is used as a stable tie-breaker.

df["content_id"] = df["content_id"].astype(str)

queue = (
    df[
        [
            "content_id",
            "client_id",
            "days_since_last_update",
            "impressions_90d",
            "baseline_score",
            "reason_code",
            "action",
        ]
    ]
    .sort_values(
        by=["baseline_score", "days_since_last_update", "impressions_90d", "content_id"],
        ascending=[False, False, False, True]
    )
    .reset_index(drop=True)
)

# Add rank
queue.insert(0, "rank", range(1, len(queue) + 1))

# ------------------------------------------------------------
# 8. Show the top 10
# ------------------------------------------------------------

print("TOP 10:")
print()

print(
    queue.head(10).to_string(index=False)
)

# ------------------------------------------------------------
# 9. Show action counts
# ------------------------------------------------------------

print()
print("=" * 70)
print("ACTION COUNTS")
print("=" * 70)

print(queue["action"].value_counts().to_string())

# ------------------------------------------------------------
# 10. Show score counts
# ------------------------------------------------------------

print()
print("=" * 70)
print("SCORE COUNTS")
print("=" * 70)

print(queue["baseline_score"].value_counts().sort_index(ascending=False).to_string())

# ------------------------------------------------------------
# 11. Write the required CSV
# ------------------------------------------------------------

OUTPUT_DIR = os.path.join(
    "/content/flyrank-ml",
    "work",
    "outputs"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_PATH = os.path.join(
    OUTPUT_DIR,
    "baseline_action_score.csv"
)

queue.to_csv(
    OUTPUT_PATH,
    index=False
)

# ------------------------------------------------------------
# 12. Confirm the file was created
# ------------------------------------------------------------

print()
print("=" * 70)
print("CSV CREATED")
print("=" * 70)

print(OUTPUT_PATH)
print()

print("Rows written:", len(queue))
print("Columns written:", len(queue.columns))

print()
print("Section 2 complete.")

SECTION 2 — BUILDING RANKED QUEUE
Staleness threshold: 20.0 days
Volume threshold: 731.0 impressions

TOP 10:

 rank           content_id         client_id  days_since_last_update  impressions_90d  baseline_score       reason_code  action
    1 content_7f116ae1f6f5 client_9400f1b21c                     301              954               5 STALE_HIGH_VOLUME REFRESH
    2 content_72496874f806 client_4ec9599fc2                     301              821               5 STALE_HIGH_VOLUME REFRESH
    3 content_cf56e2e2e282 client_7f2253d7e2                     194            61678               5 STALE_HIGH_VOLUME REFRESH
    4 content_7368877ea310 client_7f2253d7e2                     194            59472               5 STALE_HIGH_VOLUME REFRESH
    5 content_1bfaa38ff26c client_7f2253d7e2                     194            25715               5 STALE_HIGH_VOLUME REFRESH
    6 content_5feee3994adb client_7f2253d7e2                     194             7812               5 STALE_HIGH_VOLUME R

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
# ============================================================
# ML-07 — SECTION 3: Top-20 Review
# ============================================================

# Take the top 20 rows from our ranked queue
top20 = queue.head(20).copy()

reviews = []

for _, row in top20.iterrows():

    # --------------------------------------------------------
    # Action
    # --------------------------------------------------------
    action = row["action"]

    # --------------------------------------------------------
    # Reason code
    # --------------------------------------------------------
    reason_code = row["reason_code"]

    # --------------------------------------------------------
    # Confidence note
    # --------------------------------------------------------
    if row["baseline_score"] == 5:
        confidence_note = (
            "Higher baseline confidence because both observed signals "
            "are strong: the content is stale and has high recent volume."
        )

    elif row["baseline_score"] == 3:
        confidence_note = (
            "Moderate baseline confidence because staleness is strong, "
            "but the high-volume signal is not present."
        )

    elif row["baseline_score"] == 2:
        confidence_note = (
            "Lower baseline confidence because high volume is present, "
            "but the content is not classified as stale."
        )

    else:
        confidence_note = (
            "Low baseline confidence because neither signal is strong."
        )

    # --------------------------------------------------------
    # What would make the recommendation wrong?
    # --------------------------------------------------------
    what_would_make_it_wrong = (
        "The recommendation could be wrong if the content is already "
        "accurate and does not need an update, if the observed volume "
        "does not indicate an opportunity for improvement, or if the "
        "simple threshold rule does not capture important context."
    )

    # --------------------------------------------------------
    # Save the review
    # --------------------------------------------------------
    reviews.append({
        "rank": int(row["rank"]),
        "content_id": row["content_id"],
        "action": action,
        "reason_code": reason_code,
        "confidence_note": confidence_note,
        "what_would_make_it_wrong": what_would_make_it_wrong
    })


# Convert reviews to a table
top20_review = pd.DataFrame(reviews)


# ============================================================
# PRINT THE TOP-20 REVIEW
# ============================================================

print("=" * 90)
print("TOP-20 REVIEW")
print("=" * 90)

for _, row in top20_review.iterrows():

    print(f"\n{row['rank']}. {row['content_id']}")

    print(f"   ACTION: {row['action']}")

    print(f"   REASON CODE: {row['reason_code']}")

    print(f"   CONFIDENCE: {row['confidence_note']}")

    print(
        f"   WHAT WOULD MAKE IT WRONG: "
        f"{row['what_would_make_it_wrong']}"
    )


print()
print("=" * 90)
print("TOP-20 REVIEW COMPLETE")
print("=" * 90)

print("Reviewed rows:", len(top20_review))

TOP-20 REVIEW

1. content_7f116ae1f6f5
   ACTION: REFRESH
   REASON CODE: STALE_HIGH_VOLUME
   CONFIDENCE: Higher baseline confidence because both observed signals are strong: the content is stale and has high recent volume.
   WHAT WOULD MAKE IT WRONG: The recommendation could be wrong if the content is already accurate and does not need an update, if the observed volume does not indicate an opportunity for improvement, or if the simple threshold rule does not capture important context.

2. content_72496874f806
   ACTION: REFRESH
   REASON CODE: STALE_HIGH_VOLUME
   CONFIDENCE: Higher baseline confidence because both observed signals are strong: the content is stale and has high recent volume.
   WHAT WOULD MAKE IT WRONG: The recommendation could be wrong if the content is already accurate and does not need an update, if the observed volume does not indicate an opportunity for improvement, or if the simple threshold rule does not capture important context.

3. content_cf56e2e2e282
   

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [9]:
# ============================================================
# ML-07 — SECTION 4: Weak Picks + Leakage Check
# ============================================================

print("=" * 90)
print("SECTION 4 — WEAK PICKS + LEAKAGE CHECK")
print("=" * 90)

# ------------------------------------------------------------
# PART A — Find potentially weak picks
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("A) WEAK PICKS")
print("=" * 90)

# Look at the top 20 recommendations
weak_candidates = top20.copy()

# A simple skeptical check:
# A high score caused by staleness + volume can still be a weak pick
# if the content has very high volume but only moderate staleness,
# or if the recommendation is dominated by one signal.

weak_picks = weak_candidates[
    (
        (weak_candidates["baseline_score"] == 5)
        & (weak_candidates["days_since_last_update"] < 200)
    )
    |
    (
        (weak_candidates["baseline_score"] == 5)
        & (weak_candidates["impressions_90d"] < 2000)
    )
].copy()

# Keep at least a few examples if the filter finds none
if len(weak_picks) == 0:
    weak_picks = weak_candidates.head(3).copy()

# Limit the review to 5 weak candidates
weak_picks = weak_picks.head(5)

print(f"\nPotential weak picks found: {len(weak_picks)}")

for _, row in weak_picks.iterrows():

    print(f"\nRank {int(row['rank'])}: {row['content_id']}")

    print(f"Action: {row['action']}")

    print(f"Reason code: {row['reason_code']}")

    print(
        f"Why it may be weak: The rule gives it a high priority because "
        f"it has {int(row['days_since_last_update'])} days since its last "
        f"update and {int(row['impressions_90d']):,} impressions. However, "
        f"the simple rule does not know whether the content actually needs "
        f"an update."
    )

    print(
        "What could make the recommendation wrong: The content may already "
        "be accurate, intentionally evergreen, or have high traffic without "
        "having a refresh opportunity."
    )


# ------------------------------------------------------------
# PART B — Leakage check
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("B) LEAKAGE CHECK")
print("=" * 90)

# Columns actually used by the baseline rule
rule_columns = [
    "days_since_last_update",
    "impressions_90d"
]

print("\nColumns used by the baseline rule:")
for col in rule_columns:
    print(f"  - {col}")


# ------------------------------------------------------------
# Check 1: future-looking column names
# ------------------------------------------------------------

all_columns_lower = [str(c).lower() for c in df.columns]

future_keywords = [
    "future",
    "next_",
    "next30",
    "next_30",
    "next90",
    "next_90",
    "future_",
    "outcome",
    "label",
    "target"
]

future_like_columns = [
    col for col in df.columns
    if any(keyword in str(col).lower() for keyword in future_keywords)
]

print("\nPotential future/label-like columns found in dataset:")
print(future_like_columns)


# ------------------------------------------------------------
# Check 2: make sure none of those columns are used
# ------------------------------------------------------------

used_lower = {c.lower() for c in rule_columns}

leakage_columns_used = [
    c for c in future_like_columns
    if c.lower() in used_lower
]

if len(leakage_columns_used) == 0:

    print(
        "\nLEAKAGE CHECK: PASS — no future/label-like columns were used "
        "by the baseline scoring rule."
    )

else:

    print(
        "\nLEAKAGE CHECK: FAIL — a future/label-like column appears "
        "to be used by the scoring rule:"
    )

    print(leakage_columns_used)


# ------------------------------------------------------------
# Check 3: product flags
# ------------------------------------------------------------

product_flag_keywords = [
    "product_flag",
    "productflag",
    "flag",
    "recommendation",
    "recommended"
]

product_like_columns = [
    col for col in df.columns
    if any(keyword in str(col).lower() for keyword in product_flag_keywords)
]

print("\nPotential product/flag-related columns in dataset:")
print(product_like_columns)

product_columns_used = [
    c for c in product_like_columns
    if c.lower() in used_lower
]

if len(product_columns_used) == 0:

    print(
        "PRODUCT FLAG CHECK: PASS — no product flag or recommendation "
        "column was used by the scoring rule."
    )

else:

    print(
        "PRODUCT FLAG CHECK: REVIEW — the following flag-like columns "
        "appear to be used:"
    )

    print(product_columns_used)


# ------------------------------------------------------------
# Check 4: Explicit record of what the rule uses
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("BASELINE INPUTS")
print("=" * 90)

print(
    "The baseline score uses ONLY these two observed signals:"
)

print("1. days_since_last_update")
print("2. impressions_90d")

print(
    "\nThe score does NOT use a future outcome, future performance "
    "window, model prediction, or product flag."
)


# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("SECTION 4 SUMMARY")
print("=" * 90)

print(
    "Weak-pick review: completed."
)

print(
    "Leakage review: the baseline rule was constructed using only "
    "days_since_last_update and impressions_90d."
)

print(
    "These are observed data fields available for the baseline and "
    "are not future outcome labels."
)

print("\nSection 4 complete.")

SECTION 4 — WEAK PICKS + LEAKAGE CHECK

A) WEAK PICKS

Potential weak picks found: 5

Rank 1: content_7f116ae1f6f5
Action: REFRESH
Reason code: STALE_HIGH_VOLUME
Why it may be weak: The rule gives it a high priority because it has 301 days since its last update and 954 impressions. However, the simple rule does not know whether the content actually needs an update.
What could make the recommendation wrong: The content may already be accurate, intentionally evergreen, or have high traffic without having a refresh opportunity.

Rank 2: content_72496874f806
Action: REFRESH
Reason code: STALE_HIGH_VOLUME
Why it may be weak: The rule gives it a high priority because it has 301 days since its last update and 821 impressions. However, the simple rule does not know whether the content actually needs an update.
What could make the recommendation wrong: The content may already be accurate, intentionally evergreen, or have high traffic without having a refresh opportunity.

Rank 3: content_cf56e2

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.